# Fase 17B — Congelamento científico, splits e cinco clientes

Valida os insumos científicos dos três datasets, sem treinar modelos. A fase exige alvo explícito, representação numérica finita, treino/validação/teste disjuntos e exaustivos, unidade contra vazamento e partição do treino entre cinco clientes. Artefatos aprovados são copiados para a campanha V2 com SHA-256.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, os, shutil
import numpy as np
import pandas as pd

CAMPAIGN_ID = 'THESIS_OFFICIAL_CAMPAIGN_V2_20260901'
PROJECT = Path('/content/drive/MyDrive/Mestrado_Criptografia')
CAMPAIGN = PROJECT / 'OFFICIAL_CAMPAIGN_V2' / CAMPAIGN_ID
FREEZE = CAMPAIGN / '01_DATASET_FREEZE'
V1 = PROJECT / 'OFFICIAL_CAMPAIGN_V1' / '00_FROZEN_INPUTS'
SEED, NUM_CLIENTS = 42, 5
EXPECTED_SPLITS = {'train','validation','test'}
assert (CAMPAIGN / '00_CAMPAIGN_CONTROL' / 'PHASE17A_MASTER_GATE.json').exists(), 'Execute primeiro a Fase 17A.'
print('Campanha:', CAMPAIGN)

In [ ]:
def sha256_file(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def finite_numeric_frame(df, columns):
    a=df[columns].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float64)
    return a, bool(np.isfinite(a).all())

def normalized_split_name(x):
    s=str(x).strip().lower()
    return {'val':'validation','valid':'validation','dev':'validation','0':'train','1':'validation','2':'test'}.get(s,s)

def split_arrays_from_npz(path):
    z=np.load(path, allow_pickle=True)
    out={}
    for k in z.files:
        lk=k.lower()
        if 'train' in lk and 'client' not in lk: out['train']=np.asarray(z[k]).reshape(-1).astype(int)
        elif ('val' in lk or 'valid' in lk) and 'client' not in lk: out['validation']=np.asarray(z[k]).reshape(-1).astype(int)
        elif 'test' in lk and 'client' not in lk: out['test']=np.asarray(z[k]).reshape(-1).astype(int)
    return out, list(z.files)

def client_arrays_from_npz(path):
    z=np.load(path, allow_pickle=True); out=[]
    for k in z.files:
        if 'client' in k.lower(): out.append((k,np.asarray(z[k]).reshape(-1).astype(int)))
    if len(out)==1 and out[0][1].dtype==object: out=[]
    return sorted(out), list(z.files)

def validate_index_splits(splits, n):
    keys=set(splits)
    sets={k:set(map(int,v)) for k,v in splits.items()}
    in_range=all(all(0<=i<n for i in s) for s in sets.values())
    disjoint=all(not sets[a].intersection(sets[b]) for i,a in enumerate(sets) for b in list(sets)[i+1:])
    union=set().union(*sets.values()) if sets else set()
    return {'keys_ok':keys==EXPECTED_SPLITS,'in_range':in_range,'disjoint':disjoint,'exhaustive':union==set(range(n)),'counts':{k:len(v) for k,v in sets.items()}}, sets

def copy_with_hash(src, dst_dir):
    src=Path(src); dst_dir=Path(dst_dir); dst_dir.mkdir(parents=True,exist_ok=True)
    dst=dst_dir/src.name; shutil.copy2(src,dst)
    return {'source':str(src),'frozen_copy':str(dst),'sha256':sha256_file(dst),'size_bytes':dst.stat().st_size}

def locate_first(paths):
    for p in map(Path,paths):
        if p.exists(): return p
    return None

In [ ]:
# PHYSIONET — mortalidade hospitalar (In-hospital_death), RecordID como unidade independente.
d='PHYSIONET_CHALLENGE_2012'; out=FREEZE/d/'SCIENTIFIC_FREEZE'; out.mkdir(parents=True,exist_ok=True)
feature_path=locate_first([V1/d/'physionet_challenge_2012_features.csv',PROJECT/'results/mimic_challenge2012_derived_features.csv'])
split_path=locate_first([V1/d/'official_split_seed42.npz'])
client_path=locate_first([V1/d/'official_client_partition_seed42.npz',V1/d/'official_final_pairing_v1/client_partition_seed42.npz'])
initial_path=locate_first([V1/d/'official_initial_state.npy',V1/d/'official_final_pairing_v1/initial_state.npy'])
missing=[k for k,v in {'features':feature_path,'split':split_path,'clients':client_path,'initial':initial_path}.items() if v is None]
gate={'dataset':d,'target_semantics':'In-hospital mortality','target_column':'target','anti_leakage_unit':'RecordID','missing_artifacts':missing}
if not missing:
    df=pd.read_csv(feature_path); ids=df['RecordID'].astype(str) if 'RecordID' in df else pd.Series([],dtype=str)
    feature_cols=[c for c in df.columns if c not in {'RecordID','target'}]
    y=pd.to_numeric(df['target'],errors='coerce').to_numpy() if 'target' in df else np.array([])
    split,split_keys=split_arrays_from_npz(split_path); split_check,split_sets=validate_index_splits(split,len(df))
    clients,client_keys=client_arrays_from_npz(client_path); client_sets=[set(map(int,a)) for _,a in clients]
    client_union=set().union(*client_sets) if client_sets else set(); train_set=split_sets.get('train',set())
    client_disjoint=all(not client_sets[i].intersection(client_sets[j]) for i in range(len(client_sets)) for j in range(i+1,len(client_sets)))
    checks={
      'rows_positive':len(df)>0,'recordid_present':'RecordID' in df,'recordid_unique':len(ids)==len(set(ids)),
      'target_present':'target' in df,'target_binary':set(pd.Series(y).dropna().unique()).issubset({0,1}) and len(set(pd.Series(y).dropna().unique()))==2,
      'feature_count_265':len(feature_cols)==265,'split_valid':all(split_check[k] for k in ['keys_ok','in_range','disjoint','exhaustive']),
      'five_clients':len(clients)==5,'clients_disjoint':client_disjoint,'clients_cover_train_exactly':client_union==train_set,
      'initial_vector_nonempty':np.load(initial_path,allow_pickle=True).size>0
    }
    gate.update({'rows':len(df),'feature_count':len(feature_cols),'class_counts':pd.Series(y).value_counts(dropna=False).to_dict(),'split':split_check,'client_counts':{k:len(a) for k,a in clients},'checks':checks,'approved_for_training':all(checks.values())})
    gate['frozen_artifacts']=[copy_with_hash(p,out) for p in [feature_path,split_path,client_path,initial_path]] if gate['approved_for_training'] else []
else: gate['approved_for_training']=False
(out/'PHASE17B_SCIENTIFIC_GATE.json').write_text(json.dumps(gate,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
print(json.dumps(gate,indent=2,ensure_ascii=False,default=str))

In [ ]:
# DAHL RATS — valida alvo explícito e separação por animal/grupo antes de aceitar o split legado.
d='DAHL_RATS'; out=FREEZE/d/'SCIENTIFIC_FREEZE'; out.mkdir(parents=True,exist_ok=True)
feature_path=locate_first([V1/d/'dahl_derived_features.csv',PROJECT/'results/dahl_rats/baseline/dahl_derived_features.csv'])
split_path=locate_first([V1/d/'official_split_seed42.npz'])
manifest_path=locate_first([PROJECT/'results/reproducibility/dahl_rats/shared/rat_split_manifest.csv',PROJECT/'results/reproducibility/dahl_rats/baseline/split_manifest.csv'])
missing=[k for k,v in {'features':feature_path,'split':split_path,'animal_manifest':manifest_path}.items() if v is None]
gate={'dataset':d,'anti_leakage_unit':'animal/source series','missing_artifacts':missing}
if not missing:
    df=pd.read_csv(feature_path); lower={c.lower():c for c in df.columns}
    target_col=next((lower[x] for x in ['target','label','class','y'] if x in lower),None)
    group_col=next((lower[x] for x in ['animal','animal_id','rat','rat_id','source','source_file','series'] if x in lower),None)
    split,split_keys=split_arrays_from_npz(split_path); split_check,split_sets=validate_index_splits(split,len(df))
    mf=pd.read_csv(manifest_path); mlower={c.lower():c for c in mf.columns}
    mgroup=next((mlower[x] for x in ['animal','animal_id','rat','rat_id','source','source_file','series','group'] if x in mlower),None)
    msplit=next((mlower[x] for x in ['split','set','partition'] if x in mlower),None)
    leakage_free=False; manifest_splits=[]
    if mgroup and msplit:
        tmp=mf[[mgroup,msplit]].dropna().copy(); tmp[msplit]=tmp[msplit].map(normalized_split_name)
        leakage_free=bool((tmp.groupby(mgroup)[msplit].nunique()<=1).all()); manifest_splits=sorted(tmp[msplit].unique().tolist())
    y=pd.to_numeric(df[target_col],errors='coerce') if target_col else pd.Series([],dtype=float)
    feature_cols=[c for c in df.columns if c not in {target_col,group_col}]
    checks={
      'rows_positive':len(df)>0,'target_explicit':target_col is not None,
      'target_binary':target_col is not None and set(y.dropna().unique()).issubset({0,1}) and len(set(y.dropna().unique()))==2,
      'features_present':len(feature_cols)>0,'split_valid':all(split_check[k] for k in ['keys_ok','in_range','disjoint','exhaustive']),
      'animal_manifest_columns_found':bool(mgroup and msplit),'animal_leakage_absent':leakage_free,
      'manifest_has_three_splits':set(manifest_splits)==EXPECTED_SPLITS
    }
    gate.update({'rows':len(df),'columns':list(df.columns),'target_column':target_col,'group_column':group_col,'feature_count':len(feature_cols),'class_counts':y.value_counts(dropna=False).to_dict(),'split':split_check,'manifest_group_column':mgroup,'manifest_split_column':msplit,'manifest_splits':manifest_splits,'checks':checks,'approved_for_training':all(checks.values())})
    gate['frozen_artifacts']=[copy_with_hash(p,out) for p in [feature_path,split_path,manifest_path]] if gate['approved_for_training'] else []
else: gate['approved_for_training']=False
(out/'PHASE17B_SCIENTIFIC_GATE.json').write_text(json.dumps(gate,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
print(json.dumps(gate,indent=2,ensure_ascii=False,default=str))

In [ ]:
# CHEXCHONET — embeddings e labels alinhados; split obrigatório por paciente.
d='CHEXCHONET'; out=FREEZE/d/'SCIENTIFIC_FREEZE'; out.mkdir(parents=True,exist_ok=True)
embedding_path=locate_first([V1/d/'chexchonet_embeddings.npz'])
split_path=locate_first([V1/d/'official_split_original.npy',PROJECT/'results/chexchonet/representation/chexchonet_splits.npy'])
patient_manifest=locate_first([PROJECT/'results/chexchonet/freeze/chexchonet_patient_manifest.csv',PROJECT/'results/reproducibility/chexchonet/shared/chexchonet_patient_split.csv'])
image_manifest=locate_first([PROJECT/'results/chexchonet/freeze/chexchonet_image_manifest.csv',PROJECT/'results/reproducibility/chexchonet/shared/chexchonet_split_manifest.csv'])
missing=[k for k,v in {'embeddings':embedding_path,'split':split_path,'patient_manifest':patient_manifest,'image_manifest':image_manifest}.items() if v is None]
gate={'dataset':d,'target_semantics':'Gold-standard echocardiography label','anti_leakage_unit':'patient','missing_artifacts':missing}
if not missing:
    z=np.load(embedding_path,allow_pickle=True); keys=list(z.files)
    arrays={k:np.asarray(z[k]) for k in keys}
    xkey=next((k for k,a in arrays.items() if a.ndim==2 and np.issubdtype(a.dtype,np.number)),None)
    ykey=next((k for k,a in arrays.items() if a.ndim==1 and np.issubdtype(a.dtype,np.number) and xkey and len(a)==len(arrays[xkey]) and set(np.unique(a)).issubset({0,1})),None)
    X=arrays[xkey] if xkey else np.empty((0,0)); y=arrays[ykey] if ykey else np.array([])
    raw_split=np.load(split_path,allow_pickle=True); raw_split=np.asarray(raw_split).reshape(-1)
    split_labels=np.array([normalized_split_name(v) for v in raw_split])
    pm=pd.read_csv(patient_manifest); plower={c.lower():c for c in pm.columns}
    pid=next((plower[x] for x in ['patient_id','patient','subject_id','subject','pid'] if x in plower),None)
    psplit=next((plower[x] for x in ['split','set','partition'] if x in plower),None)
    leakage_free=False; patient_splits=[]
    if pid and psplit:
        tmp=pm[[pid,psplit]].dropna().copy(); tmp[psplit]=tmp[psplit].map(normalized_split_name)
        leakage_free=bool((tmp.groupby(pid)[psplit].nunique()<=1).all()); patient_splits=sorted(tmp[psplit].unique().tolist())
    checks={
      'embedding_matrix_found':xkey is not None,'target_vector_found':ykey is not None,
      'rows_aligned':len(X)>0 and len(X)==len(y)==len(split_labels),
      'embeddings_finite':X.size>0 and bool(np.isfinite(X).all()),
      'target_binary':y.size>0 and set(np.unique(y)).issubset({0,1}) and len(np.unique(y))==2,
      'split_has_three_sets':set(split_labels)==EXPECTED_SPLITS,
      'patient_manifest_columns_found':bool(pid and psplit),'patient_leakage_absent':leakage_free,
      'patient_manifest_has_three_splits':set(patient_splits)==EXPECTED_SPLITS
    }
    gate.update({'npz_keys':keys,'embedding_key':xkey,'target_key':ykey,'rows':len(X),'embedding_dimension':int(X.shape[1]) if X.ndim==2 else None,'class_counts':pd.Series(y).value_counts().to_dict(),'split_counts':pd.Series(split_labels).value_counts().to_dict(),'patient_id_column':pid,'patient_split_column':psplit,'checks':checks,'approved_for_training':all(checks.values())})
    gate['frozen_artifacts']=[copy_with_hash(p,out) for p in [embedding_path,split_path,patient_manifest,image_manifest]] if gate['approved_for_training'] else []
else: gate['approved_for_training']=False
(out/'PHASE17B_SCIENTIFIC_GATE.json').write_text(json.dumps(gate,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
print(json.dumps(gate,indent=2,ensure_ascii=False,default=str))

In [ ]:
# Gate mestre: somente três aprovações simultâneas liberam a preparação dos Baselines.
gates={}
for d in ['PHYSIONET_CHALLENGE_2012','DAHL_RATS','CHEXCHONET']:
    p=FREEZE/d/'SCIENTIFIC_FREEZE'/'PHASE17B_SCIENTIFIC_GATE.json'
    gates[d]=json.loads(p.read_text(encoding='utf-8'))
all_ok=all(g.get('approved_for_training') is True for g in gates.values())
master={
 'phase':'17B','campaign_id':CAMPAIGN_ID,'completed_at_utc':datetime.now(timezone.utc).isoformat(),
 'dataset_approvals':{d:g.get('approved_for_training',False) for d,g in gates.items()},
 'all_datasets_scientifically_frozen':all_ok,
 'training_authorized':all_ok,
 'next_authorized_step':'FASE_17C_BASELINE_SMOKE_TESTS' if all_ok else 'REPAIR_FAILED_PHASE17B_GATES',
 'failed_checks':{d:[k for k,v in g.get('checks',{}).items() if not v] for d,g in gates.items()},
 'official_matrix':[f'{d}__{s}' for d in gates for s in ['BASELINE','CKKS','HYBRID']],
}
control=CAMPAIGN/'00_CAMPAIGN_CONTROL'; (control/'PHASE17B_MASTER_GATE.json').write_text(json.dumps(master,indent=2,ensure_ascii=False),encoding='utf-8')
status_path=control/'CAMPAIGN_STATUS.json'; status=json.loads(status_path.read_text(encoding='utf-8'))
status.update({'status':'PHASE17B_COMPLETED' if all_ok else 'PHASE17B_BLOCKED','phase17b_training_authorized':all_ok,'usable_in_thesis':False})
status_path.write_text(json.dumps(status,indent=2,ensure_ascii=False),encoding='utf-8')
print('='*100); print(json.dumps(master,indent=2,ensure_ascii=False)); print('='*100)

In [ ]:
# Gera um pacote pequeno para análise e inicia o download.
from google.colab import files
export=Path('/content/PHASE17B_EVIDENCE'); shutil.rmtree(export,ignore_errors=True); export.mkdir()
shutil.copytree(CAMPAIGN/'00_CAMPAIGN_CONTROL',export/'00_CAMPAIGN_CONTROL')
for d in ['PHYSIONET_CHALLENGE_2012','DAHL_RATS','CHEXCHONET']:
    src=FREEZE/d/'SCIENTIFIC_FREEZE'; dst=export/'01_DATASET_FREEZE'/d/'SCIENTIFIC_FREEZE'; dst.mkdir(parents=True,exist_ok=True)
    for p in src.glob('*.json'): shutil.copy2(p,dst/p.name)
zip_path=shutil.make_archive('/content/PHASE17B_EVIDENCE','zip','/content','PHASE17B_EVIDENCE')
permanent=CAMPAIGN/'00_CAMPAIGN_CONTROL'/'EXPORTS'/'PHASE17B_EVIDENCE.zip'; permanent.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(zip_path,permanent)
print('Salvo em:',permanent); files.download(str(permanent))